# MoE composition - TEMPLATE

Compose several trained Qwen3.5-0.8B LoRA experts into one model and check the
routing decision.

This is a **template**: it becomes runnable once experts exist under
`michaelowusuntim6/<expert>-adapter` (or locally in `training/adapters/`).
The MoE design is one expert per prompt (no shared expert), so composition is
either a weighted adapter merge (MergeKit / PEFT) or several llama.cpp servers
behind the IR3DE router.


In [ ]:
!pip install -q transformers peft datasets mergekit 2>/dev/null || true


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE = "Qwen/Qwen3.5-0.8B"
EXPERTS = {
    "code_python": "michaelowusuntim6/code-python-adapter",
    "debug_review": "michaelowusuntim6/debug-review-adapter",
    "security": "michaelowusuntim6/security-adapter",
}
WEIGHTS = {"code_python": 0.4, "debug_review": 0.4, "security": 0.2}

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32,
                                             low_cpu_mem_usage=True,
                                             trust_remote_code=True)


## Option A - PEFT weighted adapter merge

Simplest path when the experts share a base model and target modules.


In [ ]:
peft_model = None
for name, repo in EXPERTS.items():
    if peft_model is None:
        peft_model = PeftModel.from_pretrained(model, repo, adapter_name=name)
    else:
        peft_model.load_adapter(repo, adapter_name=name)

peft_model.add_weighted_adapter(
    adapters=list(EXPERTS),
    weights=[WEIGHTS[n] for n in EXPERTS],
    adapter_name="moe",
    combination_type="linear",
)
peft_model.set_adapter("moe")
peft_model = peft_model.merge_and_unload()
peft_model.save_pretrained("merged/moe")
tokenizer.save_pretrained("merged/moe")


## Option B - MergeKit

Use this when you want task-arithmetic / TIES / DARE merging across full LoRA
adapters:

```bash
mergekit-yaml merge_config.yaml merged/moe
```


In [ ]:
prompts = [
    "Fix this Python memory leak.",
    "Why is this kernel commit vulnerable?",
    "Review this pull request for security issues.",
]

def route(prompt):
    # IR3DE placeholder: keyword routing. Replace with the real router when
    # orchestrator/router/ is implemented (docs/ARCHITECTURE.md).
    p = prompt.lower()
    if "kernel" in p or "commit" in p:
        return "linux_kernel"
    if "security" in p or "vulnerab" in p:
        return "security"
    if "python" in p or "leak" in p:
        return "code_python"
    return "debug_review"

FastLanguageModel.for_inference(peft_model) if False else peft_model.eval()
for prompt in prompts:
    expert = route(prompt)
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        out = peft_model.generate(inputs, max_new_tokens=128, do_sample=False)
    print(f"[expert={expert}] {prompt}\n  -> {tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)[:200]}\n")
